<a href="https://colab.research.google.com/github/pramana12345678/prasad-agentic-repo/blob/main/Prasad_RAG_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Access your secret keys
from google.colab import userdata
import os
from openai import OpenAI

api_key = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = api_key

client = OpenAI()

response = client.responses.create(
    model="gpt-4.1-mini",
    input="Hello from Prasad"
)

print(response.output[0].content[0].text)
print("API Key loaded", bool(api_key))

Hello Prasad! How can I assist you today?
API Key loaded True


In [ ]:
#Install the required libraries
!pip install -q --upgrade \
openai tiktoken pypdf \
langchain langchain-community \
langchain-chroma langchain-openai \
chromadb

In [ ]:
# Importing the standard Libraries
import time                           # For measuring execution time or adding delays
from datetime import datetime         # For handling timestamps and datetime operations

# ChromaDB Vector Database
import chromadb  # Chroma: a local-first vector database for storing and querying document embeddings

# OpenAI SDK
from openai import OpenAI             # Official OpenAI Python SDK (v1.x) for interacting with models like GPT-4


# Loads all PDF files from a directory and extracts text from each.
from langchain_community.document_loaders import PyPDFDirectoryLoader

# Base class representing a document in LangChain; useful for downstream chaining and processing.
from langchain_core.documents import Document

# Embeddings and Vector Store
# Generates vector embeddings using OpenAI’s embedding models (e.g., `text-embedding-3-small`)
from langchain_openai import OpenAIEmbeddings

# Integration for using Chroma as the vector store within LangChain’s ecosystem
from langchain_chroma import Chroma

# LangChain Utilities
# RecursiveCharacterTextSplitter intelligently breaks long text into smaller chunks with some overlap, preserving context.
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [ ]:
# Hide warnings/logs from Chroma
# This code mutes the following telemetry error in ChromaDB:
# ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
import logging
logging.getLogger("chromadb").setLevel(logging.CRITICAL)

In [ ]:
#Access environment variables via Colab secrets
from google.colab import userdata
import os
from openai import OpenAI

api_key = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = api_key

client = OpenAI()

response = client.responses.create(
    model="gpt-4.1-mini",
    input="Hello from Prasad"
)

In [ ]:
# Unzip the dataset containing the policy document
!unzip /content/PowerBI.zip

Archive:  /content/PowerBI.zip
  inflating: Introducing_Power_BI.pdf  


In [ ]:
#Initialize a PDF loader to store all the PDF files
pdf_folder_location = "/content/Introducing_Power_BI.pdf"  # Replace with the path to the folder containing your PDF files

# Initialize a PDF loader to load all PDF documents in the directory
pdf_loader = PyPDFDirectoryLoader(pdf_folder_location)

# Initialize a text splitter using OpenAI tokenizer
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',  # OpenAI tokenizer used by GPT-4 / embeddings
    chunk_size=512,               # max tokens per chunk
    chunk_overlap=16              # overlap between chunks
)

# Load and split documents
powerbi_chunks = pdf_loader.load_and_split(text_splitter)

In [ ]:
powerbi_collection = "powerbi_docs"


In [ ]:
#Instantiate the Open AI Embedding model
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    api_key=api_key,
    base_url="https://api.openai.com/v1",
    model="text-embedding-3-small"
)

In [ ]:
#Initialize a persistent Chroma db client
chromadb_client = chromadb.PersistentClient(
    path="/content/chroma_db"
)

In [ ]:
# Pinging the database client to check if the connection is alive
# the heartbeat method returns the current time in nanoseconds and is generally used to check if the server is alive
chromadb_client.heartbeat()

1778381658839319444

In [ ]:
# Instantiate a Chroma vector store to store and retrieve document embeddings

from langchain_chroma import Chroma

vectorstore = Chroma(
    collection_name=powerbi_collection,                 # reuse your earlier variable
    collection_metadata={"hnsw:space": "cosine"},       # similarity metric
    embedding_function=embedding_model,
    client=chromadb_client,
    persist_directory="/content/drive/MyDrive/chroma_db"  # or /content/chroma_db if temporary
)

In [ ]:
# confirming collection creation
chromadb_client.list_collections()

[Collection(name=powerbi_docs)]

In [ ]:
# Confirming database has been populated with the collection
chromadb_client.count_collections()

1

In [ ]:

import time

# Batch 500 chunks to send to the API at a time, pausing execution for 30 seconds afterward
i = 0  # Initialize the starting index for the chunks

while i < len(powerbi_chunks):
    vectorstore.add_documents(
        documents=powerbi_chunks[i:i+500],
        ids=["text_" + str(j) for j in range(i, i+500)]
    )
    i += 500
    time.sleep(30)

In [ ]:
vectorstore_persisted = Chroma(
    collection_name=powerbi_collection,                 # same collection name
    collection_metadata={"hnsw:space": "cosine"},       # similarity metric
    embedding_function=embedding_model,
    client=chromadb_client,
    persist_directory="/content/drive/MyDrive/chroma_db"  # same path used earlier
)

In [ ]:
# Define the chroma collection
collection = chromadb_client.get_collection(powerbi_collection)
# Count the number of records in the collection
collection.count()
# Inspect the first 2 records using the .peek() method
collection.peek(2)

{'ids': [],
 'embeddings': array([], dtype=float64),
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': []}

In [ ]:
# Create a retriever interface from the vector store
retriever = vectorstore_persisted.as_retriever(
    search_type = 'similarity',             # Use the default method based on semantic similarity
    search_kwargs = {'k': 5}                # Retrieve top 5 most similar chunks
)

# Define a sample user query
user_query = "What is Power BI"

In [ ]:
user_query = "What are the key features of Power BI?"

results = retriever.invoke(user_query)

In [ ]:
user_query = "How to add location in powerbi in my dashboard"

relevant_document_chunks = retriever.invoke(user_query)
len(relevant_document_chunks)
# Inspecting the first document
for document in relevant_document_chunks:
    print(document.page_content.replace("\t", " "))
    break

In [ ]:
qna_system_message = """You are a helpful AI assistant specialized in answering questions based only on the provided context.
Use only the information from the context to answer the question.
If the answer is not found in the context, say "I don't know based on the provided information."
Be concise, accurate, and include relevant details when necessary."""

In [ ]:
qna_user_message_template = """Context:
{context}

Question:
{question}

Answer:"""

In [ ]:
#Retrieving relevant documents

user_query = "What are Custom Visuals in Power BI?"

relevant_document_chunks = retriever.invoke(user_query)

context_list = [d.page_content for d in relevant_document_chunks]

context_for_query = "\n---\n".join(context_list)

prompt = [
    {'role': 'developer', 'content': qna_system_message},
    {'role': 'user', 'content': qna_user_message_template.format(
         context=context_for_query,
         question=user_query
        )
    }
]

try:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content

except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)  # Display the prediction

Custom Visuals in Power BI are unique visualizations created by developers that extend the standard visual capabilities of Power BI. They allow users to present data in innovative ways tailored to specific needs, enhancing the overall data visualization experience.


In [ ]:
%%writefile rag-chat.py

import os
import time
import chromadb

from openai import OpenAI
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter # Corrected import
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# -----------------------------
# Config
# -----------------------------
openai_api_key = os.environ["OPENAI_API_KEY"]

pdf_folder_location = "/content/PowerBI"
chroma_path = "/content/chroma_db"
powerbi_collection = "powerbi_docs"

client = OpenAI(api_key=openai_api_key)

# -----------------------------
# Load and chunk PDFs
# -----------------------------
pdf_loader = PyPDFDirectoryLoader(pdf_folder_location)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=512,
    chunk_overlap=16
)

powerbi_chunks = pdf_loader.load_and_split(text_splitter)

# -----------------------------
# Embedding model
# -----------------------------
embedding_model = OpenAIEmbeddings(
    api_key=openai_api_key,
    base_url="https://api.openai.com/v1",
    model="text-embedding-3-small"
)

# -----------------------------
# ChromaDB setup
# -----------------------------
chromadb_client = chromadb.PersistentClient(
    path=chroma_path
)

vectorstore = Chroma(
    collection_name=powerbi_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding_model,
    client=chromadb_client,
    persist_directory=chroma_path
)

# -----------------------------
# Add documents
# -----------------------------
batch_size = 500

for i in range(0, len(powerbi_chunks), batch_size):
    batch = powerbi_chunks[i:i + batch_size]
    vectorstore.add_documents(
        documents=batch,
        ids=[f"text_{j}" for j in range(i, i + len(batch))]
    )
    time.sleep(2)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# -----------------------------
# Prompts
# -----------------------------
qna_system_message = """
You are a helpful AI assistant answering questions using only the provided Power BI documentation context.
If the answer is not available in the context, say: "I don't know based on the provided documentation."
Be concise and accurate.
"""

qna_user_message_template = """
Context:
{context}

Question:
{question}

Answer:
"""

# -----------------------------
# Chat function
# -----------------------------
def ask_powerbi(question):
    relevant_document_chunks = retriever.invoke(question)

    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = "\n---\n".join(context_list)

    prompt = [
        {"role": "developer", "content": qna_system_message},
        {"role": "user", "content": qna_user_message_template.format(
            context=context_for_query,
            question=question
        )}
    ]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=prompt,
        temperature=0
    )

    return response.choices[0].message.content

# -----------------------------
# Command-line chat
# -----------------------------
print("Power BI RAG Chatbot is ready.")
print("Type 'exit' to stop.")

while True:
    user_query = input("\nAsk a Power BI question: ")

    if user_query.lower() in ["exit", "quit"]:
        print("Goodbye!")
        break

    answer = ask_powerbi(user_query)
    print("\nAnswer:")
    print(answer)

Overwriting rag-chat.py


In [ ]:
!python rag-chat.py

Power BI RAG Chatbot is ready.
Type 'exit' to stop.

Ask a Power BI question: Traceback (most recent call last):
  File "/content/rag-chat.py", line 126, in <module>
object address  : 0x7a02f754aa40
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
    user_query = input("\nAsk a Power BI question: ")

In [ ]:
!jupyter nbconvert --to html "Prasad RAG Project.ipynb"

[NbConvertApp] WARNING | pattern 'Prasad RAG Project.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--exec

In [ ]:
!ls

chroma_db  Introducing_Power_BI.pdf  PowerBI.zip  rag-chat.py  sample_data


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!jupyter nbconvert --to html "/content/drive/MyDrive/Colab Notebooks/Prasad RAG Project.ipynb"

[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/Prasad RAG Project.ipynb to html
[NbConvertApp] Writing 342885 bytes to /content/drive/MyDrive/Colab Notebooks/Prasad RAG Project.html


In [ ]:
from google.colab import files
files.download("/content/drive/MyDrive/Colab Notebooks/Prasad RAG Project.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>